# 04 · Agentic RAG

这一节参考课件 `RAG_theory` 的 Agentic RAG：
- **CRAG**：先评估检索质量，不够好就纠错/补检索
- **Adaptive-RAG**：先给 query 分流（简单走轻链路，复杂走重链路）
- **MemoRAG**：把高价值中间结果沉淀成 memory，后续优先复用

本 notebook 目标是“讲清楚控制闭环”，所以实现尽量精简直观：
- 仍然复用前面写好的 Chroma collection：`autel_annual_report_2024`
- 用 LLM 做路由/评估（可替换成规则/小模型）
- 每一步都打印 trace，便于课堂讲解

> 依赖：先跑 `01_data_02_chunk_ingest.ipynb` 写入 `data/chroma`。


In [6]:
from __future__ import annotations

import hashlib
import json
import os
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()

    for candidate in (cwd, cwd.parent):
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate

    raise FileNotFoundError("未找到项目根目录，请从 RAG_project 根目录或 notebooks 目录运行本 notebook。")



def load_project_env(project_root: Path) -> Path | None:
    for env_path in (project_root / ".env", project_root.parent / ".env"):
        if env_path.exists():
            load_dotenv(env_path, override=True)
            return env_path
    return None


PROJECT_ROOT = resolve_project_root()
ENV_FILE = load_project_env(PROJECT_ROOT)
CHROMA_DIR = PROJECT_ROOT / "data/chroma"
COLLECTION = "autel_annual_report_2024"

openai_api_key = os.getenv("OPENAI_API_KEY")
openai_base_url = os.getenv("OPENAI_BASE_URL", "https://openrouter.ai/api/v1")
embed_model = os.getenv("EMBED_MODEL", "text-embedding-3-small")
chat_model = os.getenv("CHAT_MODEL") or os.getenv("LLM_MODEL", "gpt-4o-mini")

assert CHROMA_DIR.exists(), f"找不到 Chroma 目录：{CHROMA_DIR.resolve()}（先跑 01_data_02_chunk_ingest.ipynb）"
assert openai_api_key, f"未加载 OPENAI_API_KEY（检查 {ENV_FILE or PROJECT_ROOT.parent / '.env'}）"

client_kwargs = {
    "api_key": openai_api_key,
    "base_url": openai_base_url,
}

# 当前本地 Chroma 已按 1536 维 embedding 建库；若更换 embedding 模型，请先重建 data/chroma。
emb = OpenAIEmbeddings(model=embed_model, **client_kwargs)
vs = Chroma(collection_name=COLLECTION, embedding_function=emb, persist_directory=str(CHROMA_DIR))

llm = ChatOpenAI(model=chat_model, temperature=0, **client_kwargs)

print("ready:", COLLECTION, "embed:", embed_model, "chat:", chat_model, "env:", ENV_FILE)


ready: autel_annual_report_2024 embed: text-embedding-3-small chat: openai/gpt-4o env: /Users/mengbai/Documents/AI-training/.env


In [7]:
# 工具：向量检索（带最小 trace）


def parse_json_object(raw: str, fallback: dict):
    text = (raw or "").strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.DOTALL).strip()

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        text = match.group(0)

    try:
        data = json.loads(text)
        if isinstance(data, dict):
            return data
    except Exception:
        pass

    return {**fallback, "_raw": (raw or "")[:240]}


def retrieve(query: str, k: int = 5):
    docs = vs.similarity_search(query, k=k)
    return docs


def doc_dedupe_key(doc):
    return (doc.metadata.get("chunk_id", doc.metadata.get("doc_id")), doc.page_content[:80])


def show_docs(docs, max_chars: int = 260):
    for i, d in enumerate(docs, 1):
        meta = {
            k: d.metadata.get(k)
            for k in ("type", "source_collection", "parse_source", "chunk_id", "h1", "h2", "h3")
            if k in d.metadata
        }
        print(f"[{i}]", meta)
        print(d.page_content[:max_chars].replace("\n", " "))
        print()


In [8]:
# 0) Self-RAG（最小闭环）：按需检索 + 对草稿做 critique，不支撑就补检索
# 课件要点：retrieve-on-demand + critique-and-fix

need_retrieve_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 Self-RAG 的路由器。判断回答这个问题是否需要检索外部知识库。\n"
            "只输出 JSON：{{\"need_retrieve\": 0|1, \"reason\": \"...\"}}。",
        ),
        ("human", "问题：{q}"),
    ]
)

critique_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 Self-RAG 的 critique 模块。\n"
            "给定问题、草稿答案、以及检索证据（可能为空），判断草稿是否被证据支持。\n"
            "只输出 JSON：{{\"supported\": 0|1, \"reason\": \"...\", \"rewrite\": \"...\"}}。\n"
            "- supported=0 时给一个更利于检索的 rewrite（中文）。",
        ),
        ("human", "问题：{q}\n\n草稿：{draft}\n\n证据：\n{evidence}"),
    ]
)

answer_no_ctx_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "你是助手。若缺少证据就保守回答，不要编造。"),
        ("human", "问题：{q}"),
    ]
)

answer_with_ctx_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是助手。必须基于给定证据回答；证据不足则明确说不足。",
        ),
        ("human", "问题：{q}\n\n证据：\n{evidence}"),
    ]
)


def self_rag(query: str, k: int = 5):
    print("=== Self-RAG trace ===")
    print("[query]", query)

    # step1: decide retrieve or not
    raw = llm.invoke(need_retrieve_prompt.format_messages(q=query)).content
    j = parse_json_object(raw, {"need_retrieve": 1, "reason": ""})
    need = 1 if int(j.get("need_retrieve", 1)) == 1 else 0
    reason = j.get("reason", "")
    if "_raw" in j:
        reason = f"bad_json: {j['_raw']}"

    print("[need_retrieve]", need)
    print("[reason]", reason)

    docs = []
    if need:
        docs = retrieve(query, k=k)
        print("[retrieve] k=", k)
        show_docs(docs)

    evidence = "\n\n".join(f"[{i}] {d.page_content[:450]}" for i, d in enumerate(docs, 1))

    # step2: draft
    if docs:
        draft = llm.invoke(answer_with_ctx_prompt.format_messages(q=query, evidence=evidence)).content.strip()
    else:
        draft = llm.invoke(answer_no_ctx_prompt.format_messages(q=query)).content.strip()

    print("[draft head]", draft[:220].replace("\n", " "))

    # step3: critique
    raw2 = llm.invoke(critique_prompt.format_messages(q=query, draft=draft, evidence=evidence)).content
    c = parse_json_object(raw2, {"supported": 0, "reason": "", "rewrite": ""})
    supported = 1 if int(c.get("supported", 0)) == 1 else 0
    creason = c.get("reason", "")
    rewrite = c.get("rewrite", "")
    if "_raw" in c:
        creason = f"bad_json: {c['_raw']}"

    print("[critique supported]", supported)
    print("[critique reason]", creason)

    if supported:
        return draft, docs

    # step4: fix by rewrite + retrieve + answer again
    if not rewrite:
        rewrite = query
    print("[rewrite]", rewrite)

    docs2 = retrieve(rewrite, k=k)
    print("[retrieve-2] k=", k)
    show_docs(docs2)

    evidence2 = "\n\n".join(f"[{i}] {d.page_content[:450]}" for i, d in enumerate(docs2, 1))
    final = llm.invoke(answer_with_ctx_prompt.format_messages(q=query, evidence=evidence2)).content.strip()
    print("[final head]", final[:220].replace("\n", " "))
    return final, docs2


Q0 = "道通2024年年报里，主营业务/产品线的收入结构是怎样的？"
self_rag_answer, self_rag_docs = self_rag(Q0, k=5)


=== Self-RAG trace ===
[query] 道通2024年年报里，主营业务/产品线的收入结构是怎样的？
[need_retrieve] 1
[reason] The question asks for specific financial details from Autel's 2024 annual report, which is beyond the knowledge cutoff date and requires up-to-date information from external sources.
[retrieve] k= 5
[1] {'parse_source': 'paddleocr_vl', 'chunk_id': 797, 'h2': '(一) 收入确认', 'h3': '1. 事项描述'}
相关信息披露详见财务报表附注五(33)及七(61)。   道通科技公司的营业收入主要来自于销售汽车综合诊断产品、TPMS系列产品、ADAS系列产品、汽车电子零部件和新能源充电桩等及提供相关产品的软件云服务。2024年度，道通科技公司营业收入金额为人民币393,225.64万元，其中主营业务收入为人民币388,497.45万元，占营业收入的98.80%。

[2] {'parse_source': 'paddleocr_vl', 'chunk_id': 344, 'h2': '2、收入和成本分析'}
break-word;'>37.27</td><td style='text-align: center; word-wrap: break-word;'>52.98</td><td style='text-align: center; word-wrap: break-word;'>44.66</td><td style='text-align: center; word-wrap: break-word;'>增加3.62个百分点</td></tr><tr><td colspan="7">主营业务分产品情况</t

[3] {'parse_source': 'paddleocr_vl', 'chunk_id': 1874, 'h2': '6、分部信息', 'h3': '(2). 报告分部的财务信息'}
<table border=1

In [9]:
# 1) CRAG（Corrective RAG）：先评估检索质量，再决定要不要补救

# 课件要点：Retrieval evaluator -> (refine/search) -> generate
# 这里做课堂版最小闭环：
# - evaluator 只输出 {label, reason, rewrite}
# - label in {correct, ambiguous, incorrect}

CRAGLabel = Literal["correct", "ambiguous", "incorrect"]

crag_eval_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 CRAG 的 Retrieval Evaluator。\n"
            "输入：用户问题 + 检索到的若干证据片段。\n"
            "输出 JSON：{{\"label\": \"correct|ambiguous|incorrect\", \"reason\": \"...\", \"rewrite\": \"...\"}}。\n"
            "- correct: 证据明显能支持回答\n"
            "- ambiguous: 有点相关但不够支撑，需要补检索\n"
            "- incorrect: 基本不相关，需要重写 query 再检索\n"
            "只输出 JSON，不要多余文字。",
        ),
        ("human", "问题：{q}\n\n证据：\n{evidence}"),
    ]
)

crag_rewrite_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 CRAG 的 Query Rewrite。输出 1 条更利于企业年报检索的中文查询句。不要回答问题。",
        ),
        ("human", "原始问题：{q}\n\n评估原因：{reason}\n\n建议 rewrite：{rewrite}"),
    ]
)


def crag(query: str, k: int = 5):
    print("=== CRAG trace ===")
    print("[query]", query)

    docs = retrieve(query, k=k)
    print("[retrieve-1] k=", k)
    show_docs(docs)

    evidence = "\n\n".join(f"[{i}] {d.page_content[:350]}" for i, d in enumerate(docs, 1))
    raw = llm.invoke(crag_eval_prompt.format_messages(q=query, evidence=evidence)).content

    ev = parse_json_object(raw, {"label": "ambiguous", "reason": "", "rewrite": ""})
    label = ev.get("label", "ambiguous")
    if label not in {"correct", "ambiguous", "incorrect"}:
        label = "ambiguous"
    reason = ev.get("reason", "")
    rewrite_hint = ev.get("rewrite", "")
    if "_raw" in ev:
        reason = f"bad_json: {ev['_raw']}"

    print("[eval]", label)
    print("[reason]", reason)

    if label == "correct":
        return docs

    # corrective action: rewrite -> retrieve again
    rewritten = llm.invoke(
        crag_rewrite_prompt.format_messages(q=query, reason=reason, rewrite=rewrite_hint)
    ).content.strip()
    print("[rewrite]", rewritten)

    docs2 = retrieve(rewritten, k=k)
    print("[retrieve-2] k=", k)
    show_docs(docs2)
    return docs2


Q1 = "道通2024年年报里，主营业务/产品线的收入结构是怎样的？给出相关表格或段落。"
crag_docs = crag(Q1, k=5)


=== CRAG trace ===
[query] 道通2024年年报里，主营业务/产品线的收入结构是怎样的？给出相关表格或段落。
[retrieve-1] k= 5
[1] {'parse_source': 'paddleocr_vl', 'chunk_id': 344, 'h2': '2、收入和成本分析'}
break-word;'>37.27</td><td style='text-align: center; word-wrap: break-word;'>52.98</td><td style='text-align: center; word-wrap: break-word;'>44.66</td><td style='text-align: center; word-wrap: break-word;'>增加3.62个百分点</td></tr><tr><td colspan="7">主营业务分产品情况</t

[2] {'parse_source': 'paddleocr_vl', 'chunk_id': 1874, 'h2': '6、分部信息', 'h3': '(2). 报告分部的财务信息'}
<table border=1 style='margin: auto; word-wrap: break-word;'><tr><td style='text-align: center; word-wrap: break-word;'>项目</td><td style='text-align: center; word-wrap: break-word;'>中国境内</td><td style='text-align: center; word-wrap: break-word;'>北美地区</td><td s

[3] {'parse_source': 'paddleocr_vl', 'chunk_id': 797, 'h2': '(一) 收入确认', 'h3': '1. 事项描述'}
相关信息披露详见财务报表附注五(33)及七(61)。   道通科技公司的营业收入主要来自于销售汽车综合诊断产品、TPMS系列产品、ADAS系列产品、汽车电子零部件和新能源充电桩等及提供相关产品的软件云服务。2024年度，道通科技公司营业收入金额为人民币393,225.

In [10]:
# 2) Adaptive-RAG：先判断 query 难度/类型，再决定走轻/重路径
# 课件要点：complexity-aware routing（简单 query 不要走重工作流）

Route = Literal["simple", "medium", "complex"]

route_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 Adaptive-RAG 的 router。根据问题复杂度输出 JSON：{{\"route\": \"simple|medium|complex\", \"reason\": \"...\"}}\n"
            "- simple：单事实/单跳，直接检索一次即可\n"
            "- medium：需要更好的召回覆盖（建议 multi-query 或 HyDE）\n"
            "- complex：需要多步/对比/多条件，建议 agentic（例如 CRAG + 多次补检索）\n"
            "只输出 JSON。",
        ),
        ("human", "问题：{q}"),
    ]
)

multi_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 Multi-Query 生成器。输出 4 条检索 query，每条一行，不要编号。",
        ),
        ("human", "问题：{q}"),
    ]
)

hyde_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 HyDE 模块。写一段可能出现在年报中的‘假设答案’，尽量包含可检索关键词，不要编造具体数值。",
        ),
        ("human", "问题：{q}"),
    ]
)


def adaptive_rag(query: str, k: int = 5):
    raw = llm.invoke(route_prompt.format_messages(q=query)).content
    r = parse_json_object(raw, {"route": "medium", "reason": ""})
    route = r.get("route", "medium")
    if route not in {"simple", "medium", "complex"}:
        route = "medium"
    reason = r.get("reason", "")
    if "_raw" in r:
        reason = f"bad_json: {r['_raw']}"

    print("=== Adaptive-RAG trace ===")
    print("[query]", query)
    print("[route]", route)
    print("[reason]", reason)

    if route == "simple":
        docs = retrieve(query, k=k)
        show_docs(docs)
        return docs

    if route == "medium":
        # 课堂版：Multi-Query + HyDE 二选一；这里都跑一遍让你对比
        queries = [s.strip() for s in llm.invoke(multi_prompt.format_messages(q=query)).content.splitlines() if s.strip()]
        if not queries:
            queries = [query]
        print("[multi-query]", queries)
        seen, merged = set(), []
        for q in queries:
            for d in retrieve(q, k=3):
                key = doc_dedupe_key(d)
                if key not in seen:
                    merged.append(d)
                    seen.add(key)
        print("[multi-query merged]", len(merged))
        show_docs(merged[:8])

        hypo = llm.invoke(hyde_prompt.format_messages(q=query)).content.strip()
        print("[hyde hypo head]", hypo[:200].replace("\n", " "))
        docs_h = retrieve(hypo, k=k)
        print("[hyde hits]")
        show_docs(docs_h)
        return merged[:8] if merged else docs_h

    # complex -> 走 CRAG（可替换成 LangGraph 多步 agent）
    return crag(query, k=k)


Q2 = "道通年报中，研发投入的主要方向是什么？并说明对应的业务/产品线。"
adaptive_docs = adaptive_rag(Q2, k=5)


=== Adaptive-RAG trace ===
[query] 道通年报中，研发投入的主要方向是什么？并说明对应的业务/产品线。
[route] complex
[reason] 问题涉及到从年报中提取多个研发投入方向，并需要关联到具体的业务或产品线，这需要多步检索和信息整合。
=== CRAG trace ===
[query] 道通年报中，研发投入的主要方向是什么？并说明对应的业务/产品线。
[retrieve-1] k= 5
[1] {'parse_source': 'paddleocr_vl', 'chunk_id': 107, 'h2': '3、研发投入情况表'}
<table border=1 style='margin: auto; word-wrap: break-word;'><tr><td style='text-align: center; word-wrap: break-word;'></td><td style='text-align: center; word-wrap: break-word;'>本年度</td><td style='text-align: center; word-wrap: break-word;'>上年度</td><td style

[2] {'parse_source': 'paddleocr_vl', 'chunk_id': 54, 'h2': '1、深入推进数智化转型，持续优化运营效率'}
<table border=1 style='margin: auto; word-wrap: break-word;'><tr><td style='text-align: center; word-wrap: break-word;'>业务领域</td><td style='text-align: center; word-wrap: break-word;'>项目名称</td><td style='text-align: center; word-wrap: break-word;'>项目成果</td></t

[3] {'parse_source': 'paddleocr_vl', 'chunk_id': 110, 'h2': '3、研发投入情况表'}
##### 研发投入总额较上年发生重大变化的原因



In [11]:
# 3) MemoRAG（课堂版）：把“已回答过的高价值线索/答案片段”存入 memory store
# 思路：
# - memory 是一个独立的向量库（Chroma collection），内容是：{question, answer_clue, citations}
# - 新问题先查 memory；若命中高相似，就直接复用线索并补少量检索

MEMO_COLLECTION = "autel_annual_report_2024_memo"
memo_vs = Chroma(
    collection_name=MEMO_COLLECTION,
    embedding_function=emb,
    persist_directory=str(CHROMA_DIR),
)

memo_clue_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 MemoRAG 的 memory writer。\n"
            "给定问题 + 证据片段，提炼 5-8 条可复用的 Answer Clues（要像索引关键词），不要编造数值。\n"
            "输出 JSON：{{\"clues\": [\"...\"], \"summary\": \"...\"}}。只输出 JSON。",
        ),
        ("human", "问题：{q}\n\n证据：\n{evidence}"),
    ]
)


def memo_write(question: str, docs, memo_id: str | None = None):
    evidence = "\n\n".join(d.page_content[:500] for d in docs[:5])
    raw = llm.invoke(memo_clue_prompt.format_messages(q=question, evidence=evidence)).content
    j = parse_json_object(raw, {"clues": [], "summary": raw[:500]})

    clues = j.get("clues", [])
    if not isinstance(clues, list):
        clues = []
    summary = j.get("summary", raw[:500])

    content = "\n".join(["Answer clues:"] + clues + ["", "Summary:", summary])
    meta = {"type": "memo", "source_collection": COLLECTION}

    memo_id = memo_id or f"memo-{hashlib.sha1(question.encode('utf-8')).hexdigest()[:16]}"
    existing = memo_vs.get(ids=[memo_id], include=[])
    if existing and existing.get("ids"):
        memo_vs.delete(ids=[memo_id])

    memo_vs.add_texts([content], ids=[memo_id], metadatas=[meta])
    memo_vs.persist()
    return memo_id


def memo_search(question: str, k: int = 3):
    return memo_vs.similarity_search(question, k=k)


# 先用一个问题跑 CRAG，然后把结果写入 memo
seed_q = "道通2024年年报里，主营业务/产品线的收入结构是怎样的？"
seed_docs = crag(seed_q, k=5)
mid = memo_write(seed_q, seed_docs)
print("[memo saved]", mid)

# 再问一个相近问题，先查 memo
follow_q = "道通年报里各产品线收入占比/结构怎么描述？"
mem_hits = memo_search(follow_q, k=3)
print("=== MemoRAG trace ===")
print("[query]", follow_q)
print("[memo hits]")
show_docs(mem_hits, max_chars=400)

# 如果 memo 命中，再用 memo 的线索补一次轻检索（这里用 memo 文本做 query）
if mem_hits:
    memo_query = mem_hits[0].page_content
    print("[retrieve with memo clue]")
    docs = retrieve(memo_query, k=5)
    show_docs(docs)


=== CRAG trace ===
[query] 道通2024年年报里，主营业务/产品线的收入结构是怎样的？
[retrieve-1] k= 5
[1] {'parse_source': 'paddleocr_vl', 'chunk_id': 797, 'h2': '(一) 收入确认', 'h3': '1. 事项描述'}
相关信息披露详见财务报表附注五(33)及七(61)。   道通科技公司的营业收入主要来自于销售汽车综合诊断产品、TPMS系列产品、ADAS系列产品、汽车电子零部件和新能源充电桩等及提供相关产品的软件云服务。2024年度，道通科技公司营业收入金额为人民币393,225.64万元，其中主营业务收入为人民币388,497.45万元，占营业收入的98.80%。

[2] {'parse_source': 'paddleocr_vl', 'chunk_id': 344, 'h2': '2、收入和成本分析'}
break-word;'>37.27</td><td style='text-align: center; word-wrap: break-word;'>52.98</td><td style='text-align: center; word-wrap: break-word;'>44.66</td><td style='text-align: center; word-wrap: break-word;'>增加3.62个百分点</td></tr><tr><td colspan="7">主营业务分产品情况</t

[3] {'parse_source': 'paddleocr_vl', 'chunk_id': 1874, 'h2': '6、分部信息', 'h3': '(2). 报告分部的财务信息'}
<table border=1 style='margin: auto; word-wrap: break-word;'><tr><td style='text-align: center; word-wrap: break-word;'>项目</td><td style='text-align: center; word-wrap: break-word;'>中国境内</td><td style='text-align: center; word-

/var/folders/hb/4k5shxzs4c7by7lm5s0mmgrw0000gn/T/ipykernel_65711/4044465722.py:45: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  memo_vs.persist()


[memo saved] memo-dea68d5112dc3582
=== MemoRAG trace ===
[query] 道通年报里各产品线收入占比/结构怎么描述？
[memo hits]
[1] {'type': 'memo', 'source_collection': 'autel_annual_report_2024'}
Answer clues: 道通科技主营业务收入 汽车综合诊断产品收入 TPMS系列产品收入 ADAS系列产品收入 汽车电子零部件收入 新能源充电桩收入 软件云服务收入 2024年营业收入总额  Summary: 道通科技2024年营业收入为人民币393,225.64万元，其中主营业务收入为人民币388,497.45万元，占比98.80%。主营业务包括汽车综合诊断产品、TPMS系列产品、ADAS系列产品、汽车电子零部件、新能源充电桩及相关软件云服务。

[retrieve with memo clue]
[1] {'parse_source': 'paddleocr_vl', 'chunk_id': 797, 'h2': '(一) 收入确认', 'h3': '1. 事项描述'}
相关信息披露详见财务报表附注五(33)及七(61)。   道通科技公司的营业收入主要来自于销售汽车综合诊断产品、TPMS系列产品、ADAS系列产品、汽车电子零部件和新能源充电桩等及提供相关产品的软件云服务。2024年度，道通科技公司营业收入金额为人民币393,225.64万元，其中主营业务收入为人民币388,497.45万元，占营业收入的98.80%。

[2] {'parse_source': 'paddleocr_vl', 'chunk_id': 2, 'h2': '2024年：AI深度赋能，业务实现跨越式增长'}
2024 年，是道通科技全面拥抱 AI 的关键之年。我们以 AI 重构产业价值，以生态协同共创增长，凭借 “人工智能+垂直场景” 战略，实现营收 39.32 亿元（同比增长 21%）、净利润 6.41 亿元（同比激增 258%），经营净现金流 7.48 亿元（同比增长 72%），各项指标再创历史新高。   这一成绩的背后，是 AI 驱动的三大核心业务跃升：   #### ● 能源业务：AI 赋能能源效率革命   我们深度融合电力电子与 AI 技


- **CRAG**：
  - 先检索一次
  - LLM 只负责当“裁判”：这批证据够不够
  - 不够就触发 rewrite + 补检索
- **Adaptive-RAG**：
  - 核心是 *routing*：简单问题不要走重链路
  - 这节用 `simple/medium/complex` 三档让逻辑一眼可见
- **MemoRAG**：
  - 把“已发现的线索”当作新的可检索资产
  - 后续相似问题先查 memo，减少重复检索
